<div dir="rtl">

# ۰۲ — بررسی کیفیت داده (تسک ۵ فاز اول)

**هدف:** پیش از هر تحلیلی، مشکلات احتمالی داده را پیدا کنیم و برای هر کدام **تصمیم و دلیل** ثبت کنیم.

مواردی که سند پروژه خواسته:
1. رکوردهای تکراری
2. مقادیر غیرممکن یا خارج از محدوده
3. داده‌ی گمشده — و جدا کردن «پرسیده نشده» از «جواب نداده»
4. کدهای ویژه (جواب نداد، نمی‌داند، کدهای سقف و کف)
5. ناسازگاری‌های منطقی

**دامنه‌ی بررسی:** بزرگسالان ۱۸ سال و بالاتر (۸٬۱۵۳ نفر)، یعنی جمعیتی که پروژه درباره‌اش است.

خروجی نهایی: **جدول کیفیت داده** که در گزارش فاز اول می‌آید.

</div>

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("..").resolve() / "src"))
import nhanes as nh

pd.set_option("display.width", 160)

demo = nh.load_raw("DEMO_L")
slq = nh.load_raw("SLQ_L")
smq = nh.load_raw("SMQ_L")
merged = nh.add_derived_variables(nh.merge_files(demo, slq, smq))

adults = merged[merged["RIDAGEYR"] >= nh.MIN_AGE].copy()
print(f"Adults 18+: {len(adults):,}")

Adults 18+: 8,153


<div dir="rtl">

## ۱. رکوردهای تکراری

اگر یک نفر دو بار در داده آمده باشد، دو بار شمرده می‌شود و نتایج سوگیر می‌شوند.

</div>

In [3]:
n_dup = adults["SEQN"].duplicated().sum()
print(f"Duplicated SEQN among adults: {n_dup}")

Duplicated SEQN among adults: 0


<div dir="rtl">

## ۲. مقادیر غیرممکن یا خارج از محدوده

برای هر متغیر، کمینه و بیشینه‌ی مشاهده‌شده را با محدوده‌ی مجاز طبق مستندات NHANES مقایسه می‌کنیم. کدهای ویژه (مثل ۷، ۹ و ۹۹۹) در بخش ۴ جداگانه بررسی می‌شوند.

</div>

In [4]:
allowed = {
    "RIDAGEYR": (18, 80),     # age, top-coded at 80
    "RIAGENDR": (1, 2),       # 1 male, 2 female
    "WTINT2YR": (0.0001, None),  # weights must be positive
    "SLD012": (2, 14),        # 2 = <3 h, 14 = >=14 h
    "SLD013": (2, 14),
}
rows = []
for col, (lo, hi) in allowed.items():
    x = adults[col].dropna()
    out = ((x < lo) | ((x > hi) if hi is not None else False)).sum()
    rows.append({"variable": col, "min": x.min(), "max": x.max(),
                 "allowed_min": lo, "allowed_max": hi, "n_out_of_range": int(out)})
pd.DataFrame(rows)

,variable,min,max,allowed_min,allowed_max,n_out_of_range
0,RIDAGEYR,18.000000,80.000000,18.0000,80.0,0
1,RIAGENDR,1.000000,2.000000,1.0000,2.0,0
2,WTINT2YR,4584.463196,170968.343177,0.0001,NaN,0
3,SLD012,2.000000,14.000000,2.0000,14.0,0
4,SLD013,2.000000,14.000000,2.0000,14.0,0


In [5]:
# Sleep values should be multiples of 0.5 hour
for col in ["SLD012", "SLD013"]:
    x = adults[col].dropna()
    print(col, "values not on a 0.5 h grid:", int(((x * 2) % 1 != 0).sum()))

SLD012 values not on a 0.5 h grid: 0
SLD013 values not on a 0.5 h grid: 0


<div dir="rtl">

## ۳. داده‌ی گمشده: «پرسیده نشده» یا «جواب نداده»؟

یک خانه‌ی خالی دو معنای کاملاً متفاوت می‌تواند داشته باشد:
- **پرسیده نشده (پرش طبیعی):** مثلاً `SMQ040` از کسی که گفته هرگز ۱۰۰ نخ نکشیده پرسیده نمی‌شود. این **گمشده نیست**؛ جوابش از سؤال قبلی معلوم است.
- **واقعاً گمشده:** سؤال باید پرسیده می‌شد ولی جوابی ثبت نشده.

</div>

In [6]:
miss = pd.DataFrame({
    "n_missing": adults[["RIAGENDR", "RIDAGEYR", "WTINT2YR", "SLD012", "SLD013",
                         "SMQ020", "SMQ040", "SMD650"]].isna().sum()
})
miss["pct_missing"] = (miss["n_missing"] / len(adults) * 100).round(1)
miss

,n_missing,pct_missing
RIAGENDR,0,0.0
RIDAGEYR,0,0.0
WTINT2YR,0,0.0
SLD012,113,1.4
SLD013,114,1.4
SMQ020,18,0.2
SMQ040,4910,60.2
SMD650,6978,85.6


In [7]:
# SMQ040 is only asked if SMQ020 = 1 (yes)
smq040_skip = (adults["SMQ020"] != 1) & adults["SMQ040"].isna()
smq040_truly_missing = (adults["SMQ020"] == 1) & adults["SMQ040"].isna()
print(f"SMQ040 blank because not asked (skip): {smq040_skip.sum():,}")
print(f"SMQ040 blank although it should have been asked: {smq040_truly_missing.sum():,}")

print()
sleep_any_missing = adults["SLD012"].isna() | adults["SLD013"].isna()
print(f"Adults missing weekday OR weekend sleep: {sleep_any_missing.sum()}")
print(f"  - only one of the two missing: {(adults['SLD012'].isna() ^ adults['SLD013'].isna()).sum()}")
print(f"Adults with SMQ020 blank: {adults['SMQ020'].isna().sum()}")

SMQ040 blank because not asked (skip): 4,910
SMQ040 blank although it should have been asked: 0

Adults missing weekday OR weekend sleep: 142
  - only one of the two missing: 57
Adults with SMQ020 blank: 18


<div dir="rtl">

**نتیجه:** بیشتر خانه‌های خالی `SMQ040` و `SMD650` پرش طبیعی‌اند، نه داده‌ی گمشده. داده‌ی واقعاً گمشده در متغیرهای اصلی کم است.

</div>

<div dir="rtl">

## ۴. کدهای ویژه

| متغیر | کد | معنی |
|---|---|---|
| `SMQ020` | ۷ / ۹ | جواب نداد / نمی‌داند |
| `SLD012`، `SLD013` | ۲ / ۱۴ | کمتر از ۳ ساعت / ۱۴ ساعت یا بیشتر |
| `SMD650` | ۱ / ۹۵ | یک نخ یا کمتر / ۹۵ نخ یا بیشتر |
| `SMD650` | ۷۷۷ / ۹۹۹ | جواب نداد / نمی‌داند |
| `RIDAGEYR` | ۸۰ | ۸۰ سال یا بیشتر |

</div>

In [8]:
codes = {
    "SMQ020 = 7 (refused)": (adults["SMQ020"] == 7).sum(),
    "SMQ020 = 9 (don't know)": (adults["SMQ020"] == 9).sum(),
    "SLD012 = 2 (<3 h)": (adults["SLD012"] == 2).sum(),
    "SLD012 = 14 (>=14 h)": (adults["SLD012"] == 14).sum(),
    "SLD013 = 2 (<3 h)": (adults["SLD013"] == 2).sum(),
    "SLD013 = 14 (>=14 h)": (adults["SLD013"] == 14).sum(),
    "SMD650 = 1 (<=1 cigarette)": (adults["SMD650"] == 1).sum(),
    "SMD650 = 95 (>=95)": (adults["SMD650"] == 95).sum(),
    "SMD650 = 777 (refused)": (adults["SMD650"] == 777).sum(),
    "SMD650 = 999 (don't know)": (adults["SMD650"] == 999).sum(),
    "RIDAGEYR = 80 (80 or older)": (adults["RIDAGEYR"] == 80).sum(),
}
pd.Series(codes, name="n_adults").to_frame()

,n_adults
SMQ020 = 7 (refused),7
SMQ020 = 9 (don't know),7
SLD012 = 2 (<3 h),33
SLD012 = 14 (>=14 h),37
SLD013 = 2 (<3 h),33
SLD013 = 14 (>=14 h),46
SMD650 = 1 (<=1 cigarette),90
SMD650 = 95 (>=95),2
SMD650 = 777 (refused),0
SMD650 = 999 (don't know),9


<div dir="rtl">

**مهم:** اگر `SMD650 = 999` را عدد واقعی حساب کنیم، میانگین تعداد نخ کاملاً خراب می‌شود. برای همین در `nhanes.py` متغیر تمیزشده‌ی `cigs_per_day` ساخته شده که در آن ۷۷۷ و ۹۹۹ به «گمشده» تبدیل شده‌اند.

</div>

In [9]:
print("Raw SMD650 mean (wrong, includes 999):", round(adults["SMD650"].mean(), 1))
print("Cleaned cigs_per_day mean:            ", round(adults["cigs_per_day"].mean(), 1))

Raw SMD650 mean (wrong, includes 999): 19.1
Cleaned cigs_per_day mean:             11.6


<div dir="rtl">

## ۵. ناسازگاری‌های منطقی

### ۵.۱ تعداد نخ در روز فقط باید برای سیگاری‌های فعلی ثبت شده باشد

</div>

In [10]:
pd.crosstab(adults["smoking_status"], adults["SMD650"].notna(),
            colnames=["SMD650 recorded?"], dropna=False)

SMD650 recorded?,False,True
smoking_status,,
NaN,32,0
never,4878,0
former,2053,0
current,15,1175


In [12]:
pd.crosstab(adults["SMQ040"], adults["SMD650"].notna(),
            colnames=["SMD650 recorded?"])

SMD650 recorded?,False,True
SMQ040,,
1.0,0,952
2.0,15,223
3.0,2053,0


<div dir="rtl">

- هیچ فرد «هرگز سیگاری» یا «سابق» تعداد نخ ثبت‌شده ندارد ✓
- چند نفر از گروه «بعضی روزها» (`SMQ040 = 2`) تعداد نخ ندارند. احتمالاً در ۳۰ روز گذشته اصلاً سیگار نکشیده‌اند و سؤال تعداد نخ از آن‌ها پرسیده نشده. **ناسازگاری نیست**؛ این افراد در گروه «فعلی» می‌مانند، ولی در تحلیل شدت مصرف (فاز ۴) تعداد نخشان گمشده است.

</div>

In [13]:
some_days_no_cigs = ((adults["SMQ040"] == 2) & adults["SMD650"].isna()).sum()
print(f"'Some days' smokers without SMD650: {some_days_no_cigs}")

'Some days' smokers without SMD650: 15


<div dir="rtl">

### ۵.۲ تفاوت خواب روز کاری و روز تعطیل

تفاوت معمولی (مثلاً یکی دو ساعت بیشتر خوابیدن در تعطیلات) طبیعی است. تفاوت‌های خیلی بزرگ ممکن است خطای ثبت باشد، یا واقعی (مثلاً کار شیفتی).

</div>

In [14]:
diff = adults["SLD013"] - adults["SLD012"]
print(diff.describe().round(2))
print()
for k in [4, 6, 8]:
    print(f"|weekend - weekday| >= {k} h: {(diff.abs() >= k).sum()}")

count    8011.00
mean        0.56
std         1.45
min       -10.00
25%         0.00
50%         0.00
75%         1.00
max        12.00
dtype: float64

|weekend - weekday| >= 4 h: 343
|weekend - weekday| >= 6 h: 76
|weekend - weekday| >= 8 h: 19


<div dir="rtl">

**تصمیم:** این افراد حذف نمی‌شوند. تفاوت بزرگ از نظر منطقی ممکن است (کار شیفتی، جبران کم‌خوابی در تعطیلات) و شاهدی بر خطا بودن آن نداریم. تعدادشان گزارش می‌شود.

</div>

<div dir="rtl">

## ۶. آیا افراد حذف‌شده با بقیه فرق دارند؟

اگر کسانی که به خاطر داده‌ی ناقص حذف شدند با نمونه‌ی نهایی تفاوت داشته باشند، ممکن است نمونه کمی سوگیر شود. این مقایسه فقط **توصیفی** است.

</div>

In [15]:
analytic, flow = nh.build_analytic_sample(merged)
excluded = adults[~adults["SEQN"].isin(analytic["SEQN"])]

pd.DataFrame({
    "n": [len(analytic), len(excluded)],
    "mean_age": [analytic["RIDAGEYR"].mean(), excluded["RIDAGEYR"].mean()],
    "pct_male": [(analytic["RIAGENDR"] == 1).mean() * 100, (excluded["RIAGENDR"] == 1).mean() * 100],
}, index=["included", "excluded (18+)"]).round(1)

,n,mean_age,pct_male
included,7996,52.1,44.7
excluded (18+),157,56.6,54.1


<div dir="rtl">

افراد حذف‌شده به‌طور میانگین کمی مسن‌ترند و سهم مردان در آن‌ها بیشتر است. چون تعدادشان کم است (حدود ۲ درصد بزرگسالان)، اثرشان بر نتایج احتمالاً ناچیز است، ولی در بخش محدودیت‌ها ذکر می‌شود.

</div>

<div dir="rtl">

## ۷. جدول نهایی کیفیت داده

این جدول در گزارش فاز اول می‌آید و در `reports/phase1/data_quality_table.csv` ذخیره می‌شود.

</div>

In [16]:
in_analytic = adults["SEQN"].isin(analytic["SEQN"])

quality = pd.DataFrame([
    ["Duplicated SEQN", n_dup,
     "No action", "Each participant appears once"],
    ["Out-of-range values (age, sex, weight, sleep)", 0,
     "No action", "All values within documented ranges"],
    ["Sleep hours missing (weekday or weekend)", int(sleep_any_missing.sum()),
     "Excluded from analytic sample", "Response variable cannot be computed"],
    ["SMQ020 refused (7) / don't know (9)", int(adults["SMQ020"].isin([7, 9]).sum()),
     "Excluded from analytic sample", "Smoking status cannot be determined"],
    ["SMQ020 blank", int(adults["SMQ020"].isna().sum()),
     "Excluded from analytic sample", "Smoking status cannot be determined"],
    ["SMQ040 blank because SMQ020 = no", int((adults["SMQ020"] == 2).sum()),
     "Not missing - classified as never smoker", "Skip pattern of the questionnaire"],
    ["Sleep coded 2 (<3 h) or 14 (>=14 h), analytic sample",
     int(analytic["sleep_coded_extreme"].sum()),
     "Kept and flagged; sensitivity analysis in Phase 5",
     "Valid information about very short/long sleep; removing them would bias results"],
    ["SMD650 = 777 / 999", int(adults["SMD650"].isin([777, 999]).sum()),
     "Set to missing in cigs_per_day", "Not real cigarette counts"],
    ["'Some days' smokers without SMD650", int(some_days_no_cigs),
     "Kept as current smokers; cigs_per_day missing", "Likely did not smoke in past 30 days (skip)"],
    ["|Weekend - weekday sleep| >= 8 h", int((diff.abs() >= 8).sum()),
     "Kept", "Plausible (e.g. shift work); no evidence of error"],
    ["Age top-coded at 80, analytic sample", int((analytic["RIDAGEYR"] == 80).sum()),
     "Kept", "Real age unknown above 80; noted as limitation"],
], columns=["issue", "n", "decision", "reason"])

out_dir = Path("..") / "reports" / "phase1"
out_dir.mkdir(parents=True, exist_ok=True)
quality.to_csv(out_dir / "data_quality_table.csv", index=False)
quality

,issue,n,decision,reason
0,Duplicated SEQN,0,No action,Each participant appears once
1,"Out-of-range values (age, sex, weight, sleep)",0,No action,All values within documented ranges
2,Sleep hours missing (weekday or weekend),142,Excluded from analytic sample,Response variable cannot be computed
3,SMQ020 refused (7) / don't know (9),14,Excluded from analytic sample,Smoking status cannot be determined
4,SMQ020 blank,18,Excluded from analytic sample,Smoking status cannot be determined
5,SMQ040 blank because SMQ020 = no,4878,Not missing - classified as never smoker,Skip pattern of the questionnaire
6,"Sleep coded 2 (<3 h) or 14 (>=14 h), analytic ...",104,Kept and flagged; sensitivity analysis in Phase 5,Valid information about very short/long sleep;...
7,SMD650 = 777 / 999,9,Set to missing in cigs_per_day,Not real cigarette counts
8,'Some days' smokers without SMD650,15,Kept as current smokers; cigs_per_day missing,Likely did not smoke in past 30 days (skip)
9,|Weekend - weekday sleep| >= 8 h,19,Kept,Plausible (e.g. shift work); no evidence of error


<div dir="rtl">

## جمع‌بندی

- رکورد تکراری و مقدار خارج از محدوده **نداریم**.
- بیشتر خانه‌های خالی متغیرهای سیگار **پرش طبیعی** پرسش‌نامه‌اند، نه داده‌ی گمشده.
- حذف‌ها فقط برای کسانی بود که متغیر پاسخ یا گروه سیگارشان قابل تعیین نبود.
- کدهای ۲ و ۱۴ خواب و سن ۸۰ **نگه داشته شدند** و به‌عنوان محدودیت ثبت می‌شوند.
- کدهای ۷۷۷ و ۹۹۹ تعداد نخ به گمشده تبدیل شدند.

</div>